In [1]:
import pandas as pd
import numpy as np
import glob
from sklearn.preprocessing import MinMaxScaler
from imblearn.under_sampling import RandomUnderSampler
import joblib
import os

In [2]:
day_files = glob.glob("../data/raw/*.csv")
df =pd.concat((pd.read_csv(f) for f in day_files), ignore_index=True)

In [10]:
print(df.shape)
df.head()

(2830743, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,22,166,1,1,0,0,0,0,0.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
1,60148,83,1,2,0,0,0,0,0.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
2,123,99947,1,1,48,48,48,48,48.0,0.0,...,40,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
3,123,37017,1,1,48,48,48,48,48.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
4,0,111161336,147,0,0,0,0,0,0.0,0.0,...,0,1753752.625,2123197.578,4822992,95,9463032.7,2657727.996,13600000,5700287,BENIGN


In [11]:
df.columns

Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', '

In [12]:
df.columns = df.columns.str.strip()

In [13]:
to_drop = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp']
df = df.drop(columns=to_drop,errors='ignore')

In [14]:
df.columns

Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Total Length of Fwd Packets',
       'Total Length of Bwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Min Packet Length', 'Max Packet Length', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
       'SYN Flag Co

In [15]:
df['Label'] = (df['Label'] != 'BENIGN').astype(int)

In [16]:
df['Label'].value_counts()

Label
0    2273097
1     557646
Name: count, dtype: int64

In [ ]:
inf_count = np.isinf(df).values.sum()
print(f"Inf values count:{inf_count}")

Total count of infinite values:4376


In [18]:
print("Replacing all Inf's with NaN")
df.replace([np.inf, -np.inf], np.nan, inplace=True)

Replacing all Inf's with NaN


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,22,166,1,1,0,0,0,0,0.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,0
1,60148,83,1,2,0,0,0,0,0.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,0
2,123,99947,1,1,48,48,48,48,48.0,0.0,...,40,0.000,0.000,0,0,0.0,0.000,0,0,0
3,123,37017,1,1,48,48,48,48,48.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,0
4,0,111161336,147,0,0,0,0,0,0.0,0.0,...,0,1753752.625,2123197.578,4822992,95,9463032.7,2657727.996,13600000,5700287,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2830738,53,61452,4,2,180,354,45,45,45.0,0.0,...,20,0.000,0.000,0,0,0.0,0.000,0,0,0
2830739,53,171,2,2,80,272,40,40,40.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,0
2830740,53,222,2,2,90,354,45,45,45.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,0
2830741,123,16842,1,1,48,48,48,48,48.0,0.0,...,20,0.000,0.000,0,0,0.0,0.000,0,0,0


In [19]:
before = len(df)
df.dropna(inplace=True)
after = len(df)
print(f"Dropped {before - after:,} rows")

Dropped 2,867 rows


In [20]:
features = df.columns.difference(['Label'])

for col in features:
    upper_limit = df[col].quantile(0.99)
    df[col] = np.clip(df[col], a_min=None, a_max=upper_limit)
    
print("Outliers clipped.")

Outliers clipped.


In [21]:
df.columns.value_counts()

Destination Port               1
Flow Duration                  1
Total Fwd Packets              1
Total Backward Packets         1
Total Length of Fwd Packets    1
                              ..
Idle Mean                      1
Idle Std                       1
Idle Max                       1
Idle Min                       1
Label                          1
Name: count, Length: 79, dtype: int64

In [24]:
scaler = MinMaxScaler()
df[features] = scaler.fit_transform(df[features])
print("Data scaled between 0 and 1.")

Data scaled between 0 and 1.


In [ ]:
X = df.drop('Label', axis=1)
y = df['Label']
rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)
print("New class distribution:")
print(y_resampled.value_counts())


New class distribution:
Label
0    556556
1    556556
Name: count, dtype: int64


In [28]:
num_features = X_resampled.shape[1]
# Reshape for Conv1D: (Samples, Features, 1)
X_dcnn = X_resampled.values.reshape(-1, num_features, 1)
print(f"Training data shape {X_dcnn.shape}")

Training data shape (1113112, 78, 1)


In [ ]:
os.makedirs('../data/processed', exist_ok=True)
np.save('../data/processed/X_dcnn.npy', X_dcnn)
np.save('../data/processed/y_labels.npy', y_resampled)
joblib.dump(scaler, '../data/processed/cicids_scaler.pkl')

['data/processed/cicids_scaler.pkl']